In [1]:
import json
import librosa
import os
from multiprocess import Pool
import itertools


def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)


def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
# !rm -rf synthetic-segments synthetic-done
!mkdir synthetic-segments
!mkdir synthetic-done

mkdir: cannot create directory ‘synthetic-segments’: File exists
mkdir: cannot create directory ‘synthetic-done’: File exists


In [3]:
with open('group-filenames.json') as fopen:
    filenames = json.load(fopen)
len(filenames)

4823180

In [4]:
rows = []

for no, (k, v) in enumerate(filenames.items()):
    v['filename'] = k
    rows.append((no, v))

In [5]:
from tqdm import tqdm

filenames = set()
for i in tqdm(range(len(rows))):
    if not os.path.exists(rows[i][1]['filename']):
        filenames.add(rows[i][1]['filename'])
len(filenames)

100%|██████████| 4823180/4823180 [00:11<00:00, 436996.55it/s]


635

In [6]:
import re
import soundfile as sf
from tqdm import tqdm

pattern = r"<\|([\d.]+)\|>\s*(.*?)\s*<\|([\d.]+)\|>"

def loop(rows):
    rows, _ = rows
    for r in tqdm(rows):
        i, r = r

        filename_done = f'synthetic-done/{i}.json'
        try:
            with open(filename_done) as fopen:
                json.load(fopen)
                continue
        except:
            pass
        
        text = r['segment']
        words = r['word'].replace('<|endoftext|>', '').split('<|transcribeprecise|>')[1]
        filename = r['filename']
        text = text.replace('<|endoftext|>', '').split('<|transcribe|>')[1]

        if not os.path.exists(filename):
            continue

        matches = re.findall(pattern, text)
        results = [(start, end, f'<|{start}|> {content.strip()}<|{end}|>') for start, content, end in matches]
        results_word = []
        for r in results:
            before = r[0]
            after = r[1]
            l = words.find(f'<|{before}|>')
            r_t = f'<|{after}|>'
            r = words.find(r_t)
            results_word.append((before, after, words[l: r] + r_t))

        y, sr = librosa.load(filename, sr = 16000)
        audio_filenames = []
        for no, result in enumerate(results_word):
            segment_audio_filename = f'synthetic-segments/{i}-{no}.mp3'
            before = float(result[0])
            after = float(result[1])
            y_ = y[int(before * sr): int(after * sr)]
            sf.write(segment_audio_filename, y_, 16000)
            audio_filenames.append(segment_audio_filename)

        combine_all = []
        combine_all.append({
            'mode': 'whole',
            'level': 'segment',
            'texts': [text],
            'audio_filenames': [filename],
        })
        combine_all.append({
            'mode': 'whole',
            'level': 'word',
            'texts': [words],
            'audio_filenames': [filename],
        })
        combine_all.append({
            'mode': 'streaming',
            'level': 'segment',
            'texts': [r[2] for r in results],
            'audio_filenames': audio_filenames,
        })
        combine_all.append({
            'mode': 'streaming',
            'level': 'word',
            'texts': [r[2] for r in results_word],
            'audio_filenames': audio_filenames,
        })

        with open(filename_done, 'w') as fopen:
            json.dump(combine_all, fopen)

In [7]:
# loop((rows[:10], 0))

In [8]:
multiprocessing(rows, loop, cores = 30)

  0%|          | 0/160772 [00:00<?, ?it/s]/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
  0%|          | 0/160772 [00:00<?, ?it/s]/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
  0%|          | 0/160772 [00:00<?, ?it/s]/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
  0%|          | 0/160772 [00:00<?, ?it/s]/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is re

TypeError: 'NoneType' object is not iterable

In [10]:
from glob import glob

files = glob('synthetic-done/*.json')
len(files)

4822545

In [13]:
data = []
for f in tqdm(files):
    with open(f) as fopen:
        d = json.load(fopen)
    data.extend(d)

100%|██████████| 4822545/4822545 [1:16:53<00:00, 1045.24it/s] 


In [14]:
len(data)

19290180

In [15]:
import pandas as pd

pd.DataFrame(data).to_parquet('synthetic-timestamp.parquet')

In [16]:
from datasets import Dataset

dataset = Dataset.from_list(data)

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
dataset.push_to_hub('malaysia-ai/Malaysian-STT', 'synthetic')

Uploading the dataset shards: 100%|██████████| 16/16 [01:17<00:00,  4.87s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Malaysian-STT/commit/5d26fc4d4b0810a24fcf1eac96bd5657a6c80744', commit_message='Upload dataset', commit_description='', oid='5d26fc4d4b0810a24fcf1eac96bd5657a6c80744', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Malaysian-STT', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Malaysian-STT'), pr_revision=None, pr_num=None)